# Split Datasets For all stages

Split dataset to Train/Validation/Test sets for all three stages

## A. Overview

### A.1. Stages:

#### 1. BERT
- Binary classification
- Only need `informative` label with True/False values

#### 2. Llama
- Binary classification
- Only need `informative` label with True/False values
- Selected from False positive record of stage 1 only.

#### 3. Llama
- Multi-label classification
- Need `humanitarian` labels

### A.2. Spliting steps:

#### 1. Global Test sets
- Stratified to from base data
- Random split
- 10%

#### 2. Stage 1:
- all record that do not have `humanitarian` labels

#### 3. Stage 2:
- Llama prefer quality than quantity
- `humanitarian` record only
- run all through BERT and get wrong predict and low confidence score record to train


#### 4. Stage 3:
- `humanitarian` record only
- pick an amount of record that are worst false positives though both stage 1 and 2

## B. Split STAGEs Datasets

In [41]:
from pathlib import Path
import csv
import os
import random

from sklearn.model_selection import train_test_split
import pandas as pd
import matplotlib.pyplot as plt

import configuration
from src import data_utils, setup

random.seed(setup.RANDOM_SEED)

In [42]:
dataset_path = Path("..") / "data"
disaster_path = dataset_path / "disaster"
extended_filepath = dataset_path / "extended"
splitted_filepath = dataset_path / "splitted"

out_topic_times = 20

### B.1. Load Original Datasets

In [43]:
df_disaster = pd.read_csv(dataset_path / "disaster" / data_utils.DISASTER_FILE).sample(
    frac=1, random_state=setup.RANDOM_SEED
)
df_weather = pd.read_csv(extended_filepath / data_utils.WEATHER_FILE).sample(
    frac=1, random_state=setup.RANDOM_SEED
)
df_out_topic = pd.read_csv(extended_filepath / data_utils.OUT_TOPIC_FILE).sample(
    frac=1, random_state=setup.RANDOM_SEED
)

# For STAGE 3
df_humanitarian = df_disaster[~df_disaster["humanitarian_label"].isna()]
df_humanitarian.to_csv(
    splitted_filepath / 'llama_humanitarian_sets.csv',
    index=False,
    quoting=csv.QUOTE_NONNUMERIC,
)

In [44]:
print(f"Base disaster dataset size: {df_disaster.groupby('informative').size()}")
print(
    f"Base humanitarian dataset size: {df_humanitarian.groupby('informative').size()}"
)
print(f"Base weather dataset size: {len(df_weather)}")
print(f"Base out-topic dataset size: {len(df_out_topic)}")

Base disaster dataset size: informative
False    60926
True     98804
dtype: int64
Base humanitarian dataset size: informative
False    27769
True     65733
dtype: int64
Base weather dataset size: 28643
Base out-topic dataset size: 3577546


### B.2. Global Test set

In [45]:
test_frac = 0.1
test_path = splitted_filepath / "global_test_sets.csv"
print("Splitting the Global TEST datasets...")

df_test_disaster = df_disaster.sample(frac=test_frac, random_state=setup.RANDOM_SEED)
df_test_weather = df_weather.sample(frac=test_frac, random_state=setup.RANDOM_SEED)

n_test_true_disaster = len(df_test_disaster[df_test_disaster["informative"] == True])
n_test_out_topic = n_test_true_disaster * out_topic_times - len(df_test_weather) - (len(df_test_disaster) - n_test_true_disaster)

df_test_out_topic = df_out_topic.sample(n=n_test_out_topic, random_state=setup.RANDOM_SEED)

df_test = pd.concat(
    [df_test_disaster, df_test_weather, df_test_out_topic],
    ignore_index=True,
)

df_test.to_csv(
    test_path,
    index=False,
    quoting=csv.QUOTE_NONNUMERIC,
)

# Remove the samples from the original datasets
df_test_uids = set(df_test["uid"])
df_disaster = df_disaster[~df_disaster["uid"].isin(df_test_uids)]
df_weather = df_weather[~df_weather["uid"].isin(df_test_uids)]
df_out_topic = df_out_topic[~df_out_topic["uid"].isin(df_test_uids)]

# del df_test_disaster, df_test_weather, df_test_out_topic

print(f"Global TEST set size: {len(df_test)}")
print(
    f"Disaster informative samples: {df_test[df_test['subset'] == 'disaster'].groupby('informative').size()}"
)
print(
    f"Humanitarian samples: {df_test[(df_test['subset'] == 'disaster') & (df_test['humanitarian_label'].notnull())].groupby(['informative']).size()}"
)
print(
    f"Humanitarian sub-label samples: {df_test[df_test['subset'] == 'disaster'].groupby(['humanitarian_label', 'informative']).size()}"
)
print(f"Global TEST set weather samples: {len(df_test[df_test['subset'] == 'weather'])}")
print(f"Global TEST set out-topic samples: {len(df_test[df_test['subset'] == 'out_topic'])}")

display(df_test.head())

Splitting the Global TEST datasets...
Global TEST set size: 206703
Disaster informative samples: informative
False    6130
True     9843
dtype: int64
Humanitarian samples: informative
False    2747
True     6566
dtype: int64
Humanitarian sub-label samples: humanitarian_label                      informative
affected_individuals                    False            13
                                        True             32
displaced_people_and_evacuations        True            363
infrastructure_and_utility_damage       False            39
                                        True            722
injured_or_dead_people                  False            24
                                        True            432
missing_trapped_or_found_people         False             2
                                        True             35
not_humanitarian                        False          1039
                                        True            457
not_related_or_irrelevant      

,uid,tweet_text,informative,humanitarian_label,subset
0,208755,"I live with six persons, we would like to find...",True,affected_individuals,disaster
1,13936,The final seconds of Pierson’s 1-0 win in the ...,True,NaN,disaster
2,226927,@mansionz I am standing on the beach watching ...,True,other_relevant_information,disaster
3,78148,Video: Market View: 'Frankenstorm' shutters U....,True,NaN,disaster
4,183211,2.1 Due to sporadic skirmishes in eastern D.R....,True,not_humanitarian,disaster


### B.3. STAGE 1 - BERT

In [46]:
# df_disaster
# df_weather
# df_out_topic

# Only keep the samples that are not in the humanitarian dataset for the BERT dataset
df_bert_disaster = df_disaster[~df_disaster["uid"].isin(set(df_humanitarian["uid"]) | df_test_uids)]

df_bert_weather = df_weather.sample(frac=0.5, random_state=setup.RANDOM_SEED)

# number of True samples in the BERT disaster dataset
n_bert_disaster_informative = len(
    df_bert_disaster[df_bert_disaster["informative"] == True]
)
n_bert_weather = len(df_bert_weather)
n_bert_out_topic = (
    n_bert_disaster_informative * out_topic_times
    - n_bert_weather
    - (len(df_bert_disaster) - n_bert_disaster_informative)
)
df_bert_out_topic = df_out_topic.sample(n=n_bert_out_topic, random_state=setup.RANDOM_SEED)

# Remove the samples from the original datasets
df_disaster = df_disaster[~df_disaster["uid"].isin(df_bert_disaster["uid"])]
df_weather = df_weather[~df_weather["uid"].isin(df_bert_weather["uid"])]
df_out_topic = df_out_topic[~df_out_topic["uid"].isin(df_bert_out_topic["uid"])]

df_bert = pd.concat(
    [df_bert_disaster, df_bert_weather, df_bert_out_topic],
    ignore_index=True,
)

df_bert.to_csv(
    splitted_filepath / "bert_sets.csv",
    index=False,
    quoting=csv.QUOTE_NONNUMERIC,
)

del df_bert_disaster, df_bert_weather, df_bert_out_topic

print(f"BERT set size: {len(df_bert)}")
print(
    f"Disaster informative samples: {df_bert[df_bert['subset'] == 'disaster'].groupby('informative').size()}"
)
print(f"BERT set weather samples: {len(df_bert[df_bert['subset'] == 'weather'])}")
print(f"BERT set out-topic samples: {len(df_bert[df_bert['subset'] == 'out_topic'])}")


BERT set size: 625674
Disaster informative samples: informative
False    29774
True     29794
dtype: int64
BERT set weather samples: 12890
BERT set out-topic samples: 553216


### B.4. STAGE 2 & 3 - Llama

In [47]:
# n_true_humanitarian = len(df_humanitarian[df_humanitarian["informative"] == True])
# n_llama_out_topic = n_true_humanitarian * out_topic_times - len(df_weather) - (len(df_disaster) - n_true_humanitarian)
# df_out_topic_llama = df_out_topic.sample(n=n_llama_out_topic, random_state=setup.RANDOM_SEED)

# put all the remaining samples into the LLAMA set
df_llama = pd.concat(
    [df_disaster, df_weather, df_out_topic],
    ignore_index=True,
)

print(f"LLAMA set size: {len(df_llama)}")
print(
    f"Disaster informative samples: {df_llama[df_llama['subset'] == 'disaster'].groupby('informative').size()}"
)
print(
        f"Humanitarian samples: {df_llama[(df_llama['subset'] == 'disaster') & (df_llama['humanitarian_label'].notnull())].groupby(['informative']).size()}"
    )
print(
    f"Humanitarian_label samples: {df_llama[df_llama['subset'] == 'disaster'].groupby(['humanitarian_label', 'informative']).size()}"
)
print(f"LLAMA set weather samples: {len(df_llama[df_llama['subset'] == 'weather'])}")
print(
    f"LLAMA set out-topic samples: {len(df_llama[df_llama['subset'] == 'out_topic'])}"
)

df_llama.to_csv(
    splitted_filepath / "llama_pre_bert_sets.csv",
    index=False,
    quoting=csv.QUOTE_NONNUMERIC,
)

LLAMA set size: 2933542
Disaster informative samples: informative
False    25022
True     59167
dtype: int64
Humanitarian samples: informative
False    25022
True     59167
dtype: int64
Humanitarian_label samples: humanitarian_label                      informative
affected_individuals                    False            139
                                        True             283
displaced_people_and_evacuations        True            2919
infrastructure_and_utility_damage       False            409
                                        True            6519
injured_or_dead_people                  False            184
                                        True            3862
missing_trapped_or_found_people         False             18
                                        True             435
not_humanitarian                        False           9383
                                        True            4367
not_related_or_irrelevant               False          12236
ot